# 09b · Why does exact fallback take longer?

The expanded09 experiment observed a higher raw-scan time after model evaluation than in the exact-only baseline.
This notebook tests that observation with controlled **forced-exact** executions. It is a cost diagnostic, not a new
acceptance policy, accuracy experiment or proof of application speedup. Models and gates are unchanged.

## Run this notebook

Use the same classic dedicated cluster/JDK and repository branch as09. No extra LightGBM install or training is needed:
model inference uses the existing Java runtime. The `09_e2e_source_information.ipynb` helper definitions and embedded
Java adapter are reused from this checkout; its run cells are never executed.

Run the first code cell to show widgets, then Run All. **Normally leave `benchmark_dir` and `run_dir` blank.**
The notebook selects the most recently completed09 run under your personal `output_root`, prints its path, and uses
its source08d directory, raw-data identity, JAR, saved test selection, models and empirical gates. It never selects
a run based on speed or answer quality. Set `benchmark_dir` explicitly to reproduce a specific completed09 run.
Do not put an08d directory or a ZIP in `benchmark_dir`.

Defaults retain the previous seeds/rates and deterministically select up to30 queries per seed. Each query runs six
arms for five repeats, in two fresh JVM launches per JVM mode. With three seeds and one rate that is10,800 raw scans;
allow several minutes on the full raw data. Warm-up uses20 saved calibration queries per seed for three fixed rounds.
Use `max_queries=0` to retain the entire previous09 selection only after the smaller diagnostic succeeds.

In [ ]:
import ast
import csv
import hashlib
import json
import math
import os
import platform
import shutil
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd


def diagnostic_defaults():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = ctx.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace' + root
        output = '/Workspace/Users/' + ctx.userName().get() + '/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir()) / 'synopsislake-results')
    return dict(repo_root=repo, output_root=output, benchmark_dir='', run_dir='', jar_path='',
                raw_dir='', raw_files='', math_jar='', java='', javac='', seeds='',
                max_queries='30', repeats='5', warmup_queries='20', warmup_rounds='3',
                launches='2', jvm_modes='default,shared_noinline', random_seed='20261113')


DIAG_DEFAULTS = diagnostic_defaults()
try:
    for key, value in DIAG_DEFAULTS.items():
        dbutils.widgets.text(key, value)
    diag_get = dbutils.widgets.get
except NameError:
    def diag_get(key):
        return os.environ.get(key.upper(), DIAG_DEFAULTS[key])

## Experimental controls

Every arm scans raw points and returns the exact answer, including queries the frozen gate would accept:

| Scan path | Direct exact | Evaluate A, then exact | Evaluate B, then exact |
|---|---|---|---|
| Shared scan helper | `exact_shared` | `a_shared` | `b_shared` |
| Separate source-level callsites | `exact_split` | `a_split` | `b_split` |

The primary comparison uses one shared helper. The split path is an additional source-level callsite control, not
an exact reimplementation of09. Each pair uses the same seed, query ID, merge rate and repetition. Query order and
the six-arm order are randomized. Order, predecessor arm and monotonic timestamps are retained.

Each configuration runs in a fresh JVM. Default flags permit normal optimization. `shared_noinline` adds only
`-XX:CompileCommand=dontinline,FallbackCostDiagnostic::sharedExact`, preventing the shared helper from being inlined
into separate callers. No global optimization setting is disabled. JVM launch order is randomized within each launch
round and recorded. Unsupported JVM flags fail rather than silently running another mode.

Raw points, histograms and model weights remain resident. Initialization, parity checks and raw-truth verification
are separate setup costs; the verification prepass also warms the runtime. Explicit warm-up uses calibration queries.
This is not a cold-storage experiment. No exact answer is substituted from a truth table.

## Reuse verified 09 helpers without executing its notebook

In [ ]:
def load_09_helpers(repo):
    notebook = Path(repo) / 'ml/notebooks/09_e2e_source_information.ipynb'
    content = json.loads(notebook.read_text())
    required = {'read_csv', 'write_json', 'file_hash', 'safe_path', 'bool_series',
                'canonical_rates', 'resolve_inputs', 'choose_query_ids', 'find_math_jar'}
    found, java = {}, None
    for cell in content['cells']:
        text = ''.join(cell.get('source', []))
        if cell['cell_type'] != 'code' or text.lstrip().startswith('%'):
            continue
        tree = ast.parse(text)
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name in required:
                found[node.name] = node
            if (isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'JAVA_SOURCE' for t in node.targets)):
                java = ast.literal_eval(node.value)
    if set(found) != required or not java:
        raise ValueError('Current09 notebook helpers are missing. Pull the complete experiments/e2e-source-information branch.')
    # Execute definitions only: never execute 09 widgets, training/evaluation, plotting or run cells.
    exec(compile(ast.Module(body=list(found.values()), type_ignores=[]), str(notebook), 'exec'), globals())
    return java, file_hash(notebook)


def resolve_completed_benchmark(cfg):
    if cfg['benchmark_dir']:
        selected = safe_path(cfg['benchmark_dir'])
    else:
        candidates = []
        for path in safe_path(cfg['output_root']).glob('e2e-source-information-*/run.json'):
            state = json.loads(path.read_text())
            if state.get('status') == 'complete' and state.get('study') == 'e2e-source-information':
                candidates.append((state.get('finished_utc', ''), path.parent))
        if not candidates:
            raise FileNotFoundError('Set benchmark_dir to the completed expanded09 output directory containing selected_queries.csv.')
        # Selection is by completion time only, never by speed or quality. The chosen path/hash are recorded.
        selected = max(candidates, key=lambda x: x[0])[1]
    state = json.loads((selected / 'run.json').read_text())
    if state.get('status') != 'complete' or state.get('study') != 'e2e-source-information':
        raise ValueError('benchmark_dir must be a completed09 run, not an08d run or a ZIP')
    if not state.get('query_coordinate_parser', '').startswith('pandas.read_csv default'):
        raise ValueError('The09 run predates the coordinate-parser fix; rerun09 before diagnosing.')
    return selected, state


def prepare_diagnostic(cfg, destination, state):
    base_java, notebook_hash = load_09_helpers(cfg['repo_root'])
    previous_dir, previous = resolve_completed_benchmark(cfg)
    print('Using completed09 run:', previous_dir, flush=True)
    source = safe_path(cfg['run_dir'] or previous['source_run'])
    if file_hash(source / 'run.json') != previous['source_run_sha256']:
        raise ValueError('The completed08d source run changed')
    prior = json.loads((source / 'run.json').read_text())
    if prior.get('status') != 'complete' or prior.get('study') != 'source-information':
        raise ValueError('Requires completed08d source-information artifacts')
    inputs = resolve_inputs(source, prior, dict(data_dir='', histogram_dir=''))
    for name, digest in previous['feature_code_sha256'].items():
        if file_hash(safe_path(cfg['repo_root']) / 'ml' / name) != digest:
            raise ValueError('Feature definitions changed since09: ' + name)
    chosen = read_csv(previous_dir / 'selected_queries.csv')
    keys = ['seed', 'qId', 'mergeRate']
    if chosen.empty or chosen.duplicated(keys).any():
        raise ValueError('Empty or duplicated saved selection')
    seeds = [int(x) for x in cfg['seeds'].split(',')] if cfg['seeds'] else list(previous['config']['seeds'])
    if not set(seeds) <= set(previous['config']['seeds']) or len(set(seeds)) != len(seeds):
        raise ValueError('Seeds must be a unique subset of the completed09 seeds')
    for key in ('max_queries', 'repeats', 'warmup_queries', 'warmup_rounds', 'launches', 'random_seed'):
        cfg[key] = int(cfg[key])
    if cfg['max_queries'] < 0 or min(cfg['repeats'], cfg['warmup_queries'], cfg['warmup_rounds'], cfg['launches']) < 1:
        raise ValueError('Query limit must be nonnegative; repeats/warm-up/launches must be positive')
    modes = cfg['jvm_modes'].split(',')
    if not modes or len(modes) != len(set(modes)) or not set(modes) <= {'default', 'shared_noinline'}:
        raise ValueError('jvm_modes must contain default and/or shared_noinline')
    cfg['resolved_seeds'], cfg['resolved_jvm_modes'] = seeds, modes
    for name in ('jar_path', 'raw_dir', 'raw_files', 'java', 'javac'):
        cfg[name] = cfg[name] or previous['config'][name]
    stage = Path(tempfile.mkdtemp(prefix='synopsislake-fallback-'))
    (stage / 'run').mkdir()
    (stage / 'models').mkdir()
    state['staging_directory'] = str(stage)
    for key, path in inputs.items():
        if key.startswith('run/') and key.endswith('.txt'):
            shutil.copyfile(path, stage / key)
            if file_hash(stage / key) != prior['input_sha256'][key]:
                raise ValueError('Histogram changed during staging')
    # Crucial: default parser matches the08d feature builder, not round_trip.
    bounds = pd.read_csv(inputs['data/queries-Ans.txt'], header=None)
    bounds.columns = ['minLon', 'maxLon', 'minLat', 'maxLat', 'truth']
    splits = read_csv(source / 'splits.csv')
    if splits.duplicated(['seed', 'qId']).any():
        raise ValueError('Duplicate saved split IDs')
    rates = sorted(chosen.mergeRate.unique())
    chosen = chosen[chosen.seed.isin(seeds)].copy()
    if not np.isfinite(chosen[['minLon', 'maxLon', 'minLat', 'maxLat', 'truth', 'expectedAlog', 'expectedBlog', 'expectedEstimate']].to_numpy()).all():
        raise ValueError('Nonfinite saved test cases')
    policies = previous['config']['policies']
    if 'empirical' not in policies:
        raise ValueError('This focused diagnostic requires a completed empirical09 comparison')
    gates = read_csv(source / 'gate_details.csv')
    previous_parity = read_csv(previous_dir / 'empirical/parity.csv')
    for flag in ('score_close', 'decision_match', 'estimate_match'):
        if not bool_series(previous_parity[flag]).all():
            raise ValueError('The source09 parity checks were not all successful')
    budget = int(previous['config']['resolved_tree_budget'])
    arm_paths = {}
    for path in (stage / 'run').glob('arm_*.txt'):
        rate = float(path.read_text().splitlines()[0])
        arm_paths[rate] = str(path)
    selected, warm = [], []
    model_rows, gate_rows = [], []
    for seed in seeds:
        split = splits[splits.seed == seed]
        test_ids = split.loc[split.split == 'test', 'qId'].astype(int)
        cal_ids = split.loc[split.split == 'calibration', 'qId'].astype(int)
        old = chosen[chosen.seed == seed]
        old_ids = set(choose_query_ids(test_ids, int(previous['config']['max_queries']),
                                       int(previous['config']['random_seed']) + seed))
        if set(old.qId) != old_ids or len(old) != len(old_ids) * len(rates):
            raise ValueError('Saved09 selection does not match its declared deterministic test cohort')
        sample = choose_query_ids(old_ids, cfg['max_queries'], cfg['random_seed'] + seed)
        part = old[old.qId.isin(sample)].copy()
        if not np.array_equal(part[['minLon', 'maxLon', 'minLat', 'maxLat', 'truth']].to_numpy(),
                              bounds.loc[part.qId].to_numpy()):
            raise ValueError('Saved query coordinates differ from original feature-builder parsing')
        for variant, column in [('no_sa_contrast', 'expectedAlog'), ('no_sa_all_source', 'expectedBlog')]:
            folder = source / 'models' / variant / f'trees_{budget}' / f'seed_{seed}'
            schema = json.loads((folder / 'schema.json').read_text())
            origin = [x for x in previous['models'] if x['variant'] == variant and x['seed'] == seed]
            if (len(origin) != 1 or schema['features'] != origin[0]['features'] or schema['objective'] != 'regression'
                    or {'skew_pred', 'log_alpha'} & set(schema['features'])):
                raise ValueError('Model schema changed')
            model_path = stage / 'models' / f'{variant}-{seed}.txt'
            shutil.copyfile(folder / 'model.txt', model_path)
            if file_hash(model_path) != origin[0]['sha256']:
                raise ValueError('Model identity changed since09')
            state['models'].append(dict(seed=seed, variant=variant, sha256=file_hash(model_path)))
            model_rows.append(dict(seed=seed, variant=variant, modelPath=str(model_path)))
            ref = read_csv(folder / 'test_predictions.csv')
            ref['mergeRate'] = canonical_rates(ref.mergeRate, sorted(arm_paths))
            check = part.merge(ref[['qId', 'mergeRate', 'score']], on=['qId', 'mergeRate'], how='left', validate='one_to_one')
            if check.score.isna().any() or not np.allclose(check[column], check.score, rtol=1e-8, atol=1e-10):
                raise ValueError('Expected model scores differ from saved08d predictions')
            picked = gates[(gates.seed == seed) & (gates.variant == variant) & (gates.tree_budget == budget)
                           & (gates.policy == 'empirical') & (gates.score == 'learned') & (gates.scope == 'per_rate')
                           & np.isclose(gates.eps, .25, rtol=0, atol=1e-12)].copy()
            picked['mergeRate'] = canonical_rates(picked.mergeRate, sorted(arm_paths))
            for rate in rates:
                row = picked[picked.mergeRate == rate]
                if len(row) != 1:
                    raise ValueError('Missing/duplicated frozen gate')
                row = row.iloc[0]
                none = bool_series(pd.Series([row.accept_none])).iloc[0]
                cutoff = -math.inf if none else float(row.cutoff)
                if not none and not math.isfinite(cutoff):
                    raise ValueError('Invalid finite gate cutoff')
                frozen = previous_parity[(previous_parity.seed == seed) & (previous_parity.variant == variant)
                    & np.isclose(previous_parity.mergeRate, rate, rtol=0, atol=1e-12)].cutoff_log.unique()
                if len(frozen) != 1 or float(frozen[0]) != cutoff:
                    raise ValueError('Frozen gate cutoff changed since09; no recalibration is allowed')
                gate_rows.append(dict(seed=seed, variant=variant, mergeRate=rate, cutoff_log=cutoff))
        part.armPath = [arm_paths[r] for r in part.mergeRate]
        selected.append(part)
        cal_sample = choose_query_ids(cal_ids, cfg['warmup_queries'], cfg['random_seed'] + seed)
        for rate in rates:
            for qid in cal_sample:
                warm.append(dict(seed=seed, qId=qid, mergeRate=rate, armPath=arm_paths[rate],
                                 **bounds.loc[qid].to_dict(), expectedAlog='', expectedBlog='', expectedEstimate=''))
    cases = pd.concat(selected, ignore_index=True)
    cases.to_csv(stage / 'cases.csv', index=False)
    cases.to_csv(destination / 'selected_queries.csv', index=False)
    pd.DataFrame(warm, columns=cases.columns).to_csv(stage / 'warmup.csv', index=False)
    pd.DataFrame(model_rows).to_csv(stage / 'models.csv', index=False)
    pd.DataFrame(gate_rows).to_csv(stage / 'gates.csv', index=False)
    raw_paths = [safe_path(cfg['raw_dir']) / name.strip() for name in cfg['raw_files'].split(',')]
    if len(set(raw_paths)) != len(raw_paths):
        raise ValueError('Duplicate raw files')
    state['raw_files'] = []
    previous_raw = {Path(x['path']).name: x for x in previous['raw_files']}
    for path in raw_paths:
        before = path.stat(); digest = file_hash(path); after = path.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise ValueError('Raw data changed while hashing')
        if path.name not in previous_raw or digest != previous_raw[path.name]['sha256']:
            raise ValueError('Raw data differs from the completed09 benchmark')
        state['raw_files'].append(dict(path=str(path), size=after.st_size, mtime_ns=after.st_mtime_ns, sha256=digest))
    if set(p.name for p in raw_paths) != set(previous_raw):
        raise ValueError('Raw file set differs from09')
    (stage / 'raw_files.txt').write_text(''.join(str(p) + '\n' for p in raw_paths))
    (stage / 'sources.txt').write_text(''.join(str(p) + '\n' for p in sorted((stage / 'run').glob('src_*.txt'))))
    jar = safe_path(cfg['jar_path'])
    if file_hash(jar) != previous['jar_sha256']:
        raise ValueError('Use the same runtime JAR as the09 comparison for this diagnostic')
    math_jar = find_math_jar(cfg)
    (stage / 'SourceInformationBenchmark.java').write_text(base_java)
    (stage / 'FallbackCostDiagnostic.java').write_text(FALLBACK_JAVA)
    state.update(benchmark_dir=str(previous_dir), benchmark_manifest_sha256=file_hash(previous_dir / 'run.json'),
                 source_run=str(source), source_manifest_sha256=file_hash(source / 'run.json'),
                 source09_notebook_sha256=notebook_hash, jar_sha256=file_hash(jar), math_jar_sha256=file_hash(math_jar),
                 selected_query_cases=len(cases), selected_unique_qids=int(cases.qId.nunique()),
                 current_training_seconds=0.0, current_calibration_seconds=0.0,
                 query_coordinate_parser='pandas.read_csv default; same as08d/09',
                 diagnostic_adapter_sha256=file_hash(stage / 'FallbackCostDiagnostic.java'))
    print('Diagnostic query-seed cases:', len(cases), '| measured requests:',
          len(cases) * 6 * cfg['repeats'] * len(modes) * cfg['launches'], flush=True)
    return stage, jar, math_jar

## Execute independent JVM launches

In [ ]:
def execute_diagnostic(cfg, destination, state, prepared):
    stage, jar, math_jar = prepared
    cp = os.pathsep.join([str(stage), str(jar), str(math_jar)])
    compiled = subprocess.run([cfg['javac'], '-encoding', 'UTF-8', '-source', '8', '-target', '8', '-cp', cp,
                              str(stage / 'SourceInformationBenchmark.java'), str(stage / 'FallbackCostDiagnostic.java')],
                             capture_output=True, text=True)
    (destination / 'compile.log').write_text(compiled.stdout + compiled.stderr)
    if compiled.returncode:
        raise RuntimeError('Compilation failed; inspect compile.log. Use a classic dedicated cluster with a JDK.')
    rng = np.random.default_rng(cfg['random_seed'])
    state['launch_order'] = []
    for launch in range(cfg['launches']):
        for mode in rng.permutation(cfg['resolved_jvm_modes']):
            out = destination / f'launch_{launch}_{mode}'
            out.mkdir()
            props = dict(cases=stage / 'cases.csv', warmup=stage / 'warmup.csv', models=stage / 'models.csv',
                         gates=stage / 'gates.csv', sources=stage / 'sources.txt', raw_files=stage / 'raw_files.txt',
                         output=out, repeats=cfg['repeats'], warmup_rounds=cfg['warmup_rounds'],
                         random_seed=cfg['random_seed'] + launch, launch_id=launch, jvm_mode=mode)
            prop_path = stage / f'{out.name}.properties'
            prop_path.write_text(''.join(f'{k}={v}\n' for k, v in props.items()))
            flags = ['-XX:CompileCommand=dontinline,FallbackCostDiagnostic::sharedExact'] if mode == 'shared_noinline' else []
            cmd = [cfg['java'], *flags, '-cp', cp, 'FallbackCostDiagnostic', str(prop_path)]
            state['launch_order'].append(dict(launch=launch, jvm_mode=mode, command=cmd))
            write_json(destination / 'run.json', state)
            with (out / 'execution.log').open('w') as log:
                print('Running', out.name, flush=True)
                process = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
                started, reported = time.perf_counter(), 0
                try:
                    while process.poll() is None:
                        time.sleep(1)
                        elapsed = int(time.perf_counter() - started)
                        if elapsed // 30 > reported:
                            reported = elapsed // 30
                            print(out.name, 'elapsed seconds:', elapsed, flush=True)
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill(); process.wait(timeout=10)
                    raise
                if process.returncode:
                    raise RuntimeError('Java diagnostic failed: ' + (out / 'execution.log').read_text()[-5000:])
            runtime = (out / 'runtime.properties').read_text()
            if mode == 'shared_noinline' and 'dontinline' not in runtime:
                raise RuntimeError('Scoped JVM no-inline flag was not recorded')
            print('Completed', out.name, flush=True)
    for row in state['raw_files']:
        stat = Path(row['path']).stat()
        if (stat.st_size, stat.st_mtime_ns) != (row['size'], row['mtime_ns']):
            raise ValueError('Raw data changed during diagnosis')

## Analyze paired costs, keeping affected observations

In [ ]:
def summarize_diagnostic(destination, cfg, state):
    folders = sorted(destination.glob('launch_*'))
    measurements = pd.concat([read_csv(p / 'measurements.csv') for p in folders], ignore_index=True)
    pair = ['launch_id', 'jvm_mode', 'seed', 'mergeRate', 'qId', 'repeat']
    arms = {'exact_shared', 'a_shared', 'b_shared', 'exact_split', 'a_split', 'b_split'}
    if set(measurements.arm) != arms or measurements.duplicated(pair + ['arm']).any():
        raise ValueError('Unexpected or duplicate arms')
    if not measurements.groupby(pair).size().eq(6).all():
        raise ValueError('Incomplete matched observations')
    if not measurements.groupby(['launch_id', 'jvm_mode', 'seed', 'mergeRate', 'qId', 'arm']).size().eq(cfg['repeats']).all():
        raise ValueError('Incomplete repetition blocks')
    if not (measurements.answer == measurements.truth).all():
        raise ValueError('A forced-exact arm did not return raw truth')
    if not (measurements.total_ns >= measurements.prefix_ns + measurements.exact_ns).all():
        raise ValueError('Invalid stage accounting')
    cpu = measurements.exact_thread_cpu_ns
    measurements['exact_cpu_available'] = cpu >= 0
    measurements['gc_observed'] = measurements.gc_count_delta > 0
    measurements['compilation_observed'] = measurements.compilation_time_ms_delta > 0
    measurements.to_csv(destination / 'measurements.csv', index=False)
    keys = ['launch_id', 'jvm_mode', 'seed', 'mergeRate', 'arm']
    rows = []
    for setting, part in measurements.groupby(keys, sort=True):
        row = dict(zip(keys, setting)); row.update(observations=len(part), query_count=part.qId.nunique(),
            gc_observations=int(part.gc_observed.sum()), compilation_observations=int(part.compilation_observed.sum()))
        for field in ('prefix_ns', 'exact_ns', 'total_ns'):
            row[field.replace('_ns', '_mean_ms')] = part[field].mean() / 1e6
            row[field.replace('_ns', '_p95_ms')] = part[field].quantile(.95) / 1e6
        valid = part[part.exact_cpu_available]
        row['exact_thread_cpu_mean_ms'] = valid.exact_thread_cpu_ns.mean() / 1e6 if len(valid) else np.nan
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(destination / 'summary.csv', index=False)
    deltas = []
    for path in ('shared', 'split'):
        base = measurements[measurements.arm == 'exact_' + path][pair + ['exact_ns', 'total_ns', 'exact_thread_cpu_ns']].rename(
            columns={'exact_ns': 'baseline_exact_ns', 'total_ns': 'baseline_total_ns', 'exact_thread_cpu_ns': 'baseline_cpu_ns'})
        for letter, variant in [('a', 'no_sa_contrast'), ('b', 'no_sa_all_source')]:
            model = measurements[measurements.arm == letter + '_' + path]
            joined = model.merge(base, on=pair, validate='one_to_one')
            joined['exact_delta_ms'] = (joined.exact_ns - joined.baseline_exact_ns) / 1e6
            joined['total_delta_ms'] = (joined.total_ns - joined.baseline_total_ns) / 1e6
            joined['cpu_delta_ms'] = np.where((joined.exact_thread_cpu_ns >= 0) & (joined.baseline_cpu_ns >= 0),
                (joined.exact_thread_cpu_ns - joined.baseline_cpu_ns) / 1e6, np.nan)
            joined['variant'], joined['scan_path'] = variant, path
            joined['would_accept'] = bool_series(joined.would_accept)
            deltas.append(joined)
    paired = pd.concat(deltas, ignore_index=True)
    paired.to_csv(destination / 'paired_deltas.csv', index=False)
    result = []
    setting = ['launch_id', 'jvm_mode', 'seed', 'mergeRate', 'variant', 'scan_path']
    for identity, part in paired.groupby(setting, sort=True):
        for label, group in [('all_queries', part), ('originally_rejected', part[~part.would_accept])]:
            result.append(dict(zip(setting, identity), subset=label, query_count=group.qId.nunique(), observations=len(group),
                mean_exact_delta_ms=group.exact_delta_ms.mean(), median_exact_delta_ms=group.exact_delta_ms.median(),
                mean_total_delta_ms=group.total_delta_ms.mean(), mean_cpu_delta_ms=group.cpu_delta_ms.mean(),
                gc_observations=int(group.gc_observed.sum()), compilation_observations=int(group.compilation_observed.sum())))
    contrasts = pd.DataFrame(result)
    contrasts.to_csv(destination / 'paired_summary.csv', index=False)
    # Block/order tables retain GC/JIT-affected observations, rather than selecting a clean-looking subset.
    measurements.groupby(keys + ['repeat'], as_index=False).agg(
        exact_mean_ns=('exact_ns', 'mean'), total_mean_ns=('total_ns', 'mean'),
        gc_observations=('gc_observed', 'sum'), compilation_observations=('compilation_observed', 'sum')).to_csv(
            destination / 'blocks.csv', index=False)
    measurements.groupby(keys + ['order', 'previous_arm'], as_index=False, dropna=False).agg(
        observations=('qId', 'size'), exact_mean_ns=('exact_ns', 'mean')).to_csv(destination / 'order_effects.csv', index=False)
    setups = []
    for path in folders:
        table = read_csv(path / 'setup.csv'); table['launch_folder'] = path.name; setups.append(table)
    pd.concat(setups, ignore_index=True).to_csv(destination / 'setup.csv', index=False)
    return summary, contrasts


def export_diagnostic(destination):
    with ZipFile(destination / 'fallback_diagnostic_results.zip', 'w', ZIP_DEFLATED) as archive:
        for path in destination.rglob('*'):
            if path.is_file() and path.suffix in {'.csv', '.json', '.md', '.log', '.properties', '.png'}:
                archive.write(path, path.relative_to(destination))


def run_fallback_diagnostic(cfg):
    destination = Path(cfg['output_root']).expanduser().resolve() / ('fallback-cost-' + uuid4().hex[:12])
    destination.mkdir(parents=True, exist_ok=False)
    state = dict(status='running', study='fallback-cost-diagnostic', config=cfg, models=[],
                 started_utc=datetime.now(timezone.utc).isoformat(), python=platform.python_version(),
                 forced_exact=True, training_seconds=0.0, calibration_seconds=0.0,
                 notes=['Every arm executes an actual raw scan, including cases the frozen model would accept.',
                        'Resident inputs; setup/parity/raw validation precede timed requests and warm the runtime.',
                        'Primary outputs retain all GC/JIT-affected observations.',
                        'Queries repeat across launches/blocks/seeds; no independent-sample inference is made.',
                        'A shared Java helper can inline under default flags; scoped dontinline is a separate JVM control.',
                        'A persistent penalty does not identify CPU cache causality.'])
    started = time.perf_counter()
    def write_manifest():
        path = destination / 'run.json'
        temporary = path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(state, indent=2, allow_nan=False, default=str))
        temporary.replace(path)
    write_manifest()
    try:
        tick = time.perf_counter(); prepared = prepare_diagnostic(cfg, destination, state)
        state['preparation_seconds'] = time.perf_counter() - tick
        write_manifest(); execute_diagnostic(cfg, destination, state, prepared)
        summary, contrasts = summarize_diagnostic(destination, cfg, state)
        state.update(status='complete', wall_seconds=time.perf_counter() - started, finished_utc=datetime.now(timezone.utc).isoformat())
        write_manifest()
        (destination / 'analysis_en.md').write_text(
            '# Fallback cost diagnostic\n\nAll arms return exact raw-data counts. These are forced-exact cost controls, '
            'not new routing policies or accuracy results. Primary comparison: model-before-scan versus direct scan '
            'on identical query/repetition pairs, within each launch, seed, rate, JVM mode and scan path.\n\n'
            '`paired_summary.csv` separates all queries from the originally rejected subset. Positive exact deltas '
            'mean the scan itself took longer after model evaluation. Compare default JVM and scoped no-inline '
            'launches, shared and split scan paths, blocks and predecessor/order effects. GC/JIT observations remain '
            'included. Thread CPU time is undefined when unavailable, not zero. CPU timing/counter instrumentation '
            'adds overhead, so total wall time is not directly interchangeable with09 production E2E timing.\n\n'
            'If a penalty disappears under a shared non-inlined helper, this suggests runtime-path sensitivity; it '
            'does not alone identify an inlining bug. If it persists, cache/branch prediction/JVM effects remain '
            'hypotheses requiring profiling. Do not claim a CPU-cache cause or an application speedup from this '
            'diagnostic. Repetitions, JVM launches and overlapping query IDs are not independent workload samples.\n')
        export_diagnostic(destination)
        print('COMPLETE:', destination, flush=True)
        return destination, summary, contrasts
    except BaseException as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}', wall_seconds=time.perf_counter() - started)
        write_manifest(); print('FAILED; diagnostics preserved:', destination, flush=True); raise

## Diagnostic Java adapter

The adapter compiles against the same JAR as the completed09 run and the unchanged09 helper source.
It records prefix, estimate, feature, inference, decision, raw-scan and total wall times. Scan thread CPU time is
unavailable when unsupported. GC/JVM compilation counters are sampled outside request timing; CPU counters bracket
the scan wall timer. Instrumentation adds overhead: do not substitute these totals for production09 latency.

In [ ]:
FALLBACK_JAVA = r"""
import java.io.*;
import java.lang.management.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;

/** Forced-exact cost diagnostic. Does not train, recalibrate or return synopsis answers. */
public final class FallbackCostDiagnostic {
    static final String A = SourceInformationBenchmark.A;
    static final String B = SourceInformationBenchmark.B;
    static final String[] ARMS = {"exact_shared", "a_shared", "b_shared", "exact_split", "a_split", "b_split"};
    static final ThreadMXBean THREAD = ManagementFactory.getThreadMXBean();
    static final CompilationMXBean COMPILATION = ManagementFactory.getCompilationMXBean();
    static final List<GarbageCollectorMXBean> COLLECTORS = ManagementFactory.getGarbageCollectorMXBeans();
    static boolean cpuEnabled;
    static long measurementStart;

    static class Result {
        long start, prefix, estimate, features, inference, decision, exact, cpu, total;
        long gcCount, gcMillis, compilationMillis;
        double answer, estimateValue = Double.NaN, score = Double.NaN, cutoff = Double.NaN;
        boolean wouldAccept;
    }

    static class Snapshot {
        long count, millis, compilation;
        Snapshot() {
            boolean anyCount = false, anyTime = false;
            for (GarbageCollectorMXBean bean : COLLECTORS) {
                long c = bean.getCollectionCount(), t = bean.getCollectionTime();
                if (c >= 0) { count += c; anyCount = true; }
                if (t >= 0) { millis += t; anyTime = true; }
            }
            if (!anyCount) count = -1;
            if (!anyTime) millis = -1;
            compilation = COMPILATION != null && COMPILATION.isCompilationTimeMonitoringSupported()
                    ? COMPILATION.getTotalCompilationTime() : -1;
        }
    }

    static long delta(long after, long before) {
        return before < 0 || after < 0 ? -1 : after - before;
    }

    static String variant(String arm) {
        return arm.startsWith("a_") ? A : arm.startsWith("b_") ? B : "none";
    }

    /** Use -XX:CompileCommand=dontinline,FallbackCostDiagnostic::sharedExact to retain this one scan callsite. */
    static double sharedExact(SourceInformationBenchmark.Query q) {
        return SourceInformationBenchmark.raw.count(q.x0, q.x1, q.y0, q.y1);
    }

    /** Control: deliberately keep three source-level scan callsites, like separate baseline/fallback branches. */
    static double splitExact(SourceInformationBenchmark.Query q, String arm) {
        if (arm.equals("exact_split"))
            return SourceInformationBenchmark.raw.count(q.x0, q.x1, q.y0, q.y1);
        if (arm.equals("a_split"))
            return SourceInformationBenchmark.raw.count(q.x0, q.x1, q.y0, q.y1);
        if (arm.equals("b_split"))
            return SourceInformationBenchmark.raw.count(q.x0, q.x1, q.y0, q.y1);
        throw new IllegalArgumentException("Unknown split arm " + arm);
    }

    static void prefix(SourceInformationBenchmark.Query q, String arm, Result r) {
        String v = variant(arm);
        if (v.equals("none")) return;
        long start = System.nanoTime();
        SourceInformationBenchmark.Hist h = SourceInformationBenchmark.resident.get(q.histKey());
        SourceInformationBenchmark.Model m = SourceInformationBenchmark.models.get(q.seed + ":" + v);
        long t = System.nanoTime();
        r.estimateValue = h.histogram.RangeQuery(q.x0, q.x1, q.y0, q.y1);
        r.estimate = System.nanoTime() - t;
        t = System.nanoTime();
        double[] x = SourceInformationBenchmark.values(m, h, q);
        r.features = System.nanoTime() - t;
        t = System.nanoTime();
        r.score = m.booster.predict(x);
        double error = Math.exp(r.score);
        r.inference = System.nanoTime() - t;
        t = System.nanoTime();
        r.cutoff = SourceInformationBenchmark.cut(q, v);
        r.wouldAccept = Double.isFinite(r.score) && error <= SourceInformationBenchmark.transformedCut(r.cutoff);
        r.decision = System.nanoTime() - t;
        r.prefix = System.nanoTime() - start;
    }

    static Result execute(SourceInformationBenchmark.Query q, String arm) {
        Snapshot before = new Snapshot();
        long start = System.nanoTime();
        Result r = new Result();
        r.start = start - measurementStart;
        prefix(q, arm, r);
        // CPU samples bracket the wall samples; neither management call is inside exact_ns.
        long cpu0 = cpuEnabled ? THREAD.getCurrentThreadCpuTime() : -1;
        long exact0 = System.nanoTime();
        r.answer = arm.endsWith("_shared") ? sharedExact(q) : splitExact(q, arm);
        r.exact = System.nanoTime() - exact0;
        long cpu1 = cpuEnabled ? THREAD.getCurrentThreadCpuTime() : -1;
        r.cpu = delta(cpu1, cpu0);
        r.total = System.nanoTime() - start;
        Snapshot after = new Snapshot();
        r.gcCount = delta(after.count, before.count);
        r.gcMillis = delta(after.millis, before.millis);
        r.compilationMillis = delta(after.compilation, before.compilation);
        return r;
    }

    static void validateResult(SourceInformationBenchmark.Query q, String arm, Result r) {
        if (r.answer != q.truth) throw new IllegalStateException("Raw count mismatch qId=" + q.id);
        String v = variant(arm);
        if (!v.equals("none") && Double.isFinite(q.expected(v))
                && r.wouldAccept != (q.expected(v) <= SourceInformationBenchmark.cut(q, v)))
            throw new IllegalStateException("Saved gate decision changed qId=" + q.id + " variant=" + v);
        if (r.total < r.prefix + r.exact)
            throw new IllegalStateException("Timing accounting failure");
    }

    static void initialize(Properties cfg, Path output, List<SourceInformationBenchmark.Query> cases,
                           List<SourceInformationBenchmark.Query> warmup) throws Exception {
        SourceInformationBenchmark.sources = SourceInformationBenchmark.paths(cfg.getProperty("sources"));
        long t = System.nanoTime();
        SourceInformationBenchmark.raw = SourceInformationBenchmark.loadRaw(
                SourceInformationBenchmark.paths(cfg.getProperty("raw_files")));
        if (SourceInformationBenchmark.raw.size() == 0) throw new IllegalArgumentException("Empty raw points");
        SourceInformationBenchmark.setupRow("raw_load", -1, "all", Double.NaN, "resident", System.nanoTime()-t,
                SourceInformationBenchmark.raw.size()+" raw points; finite validation included");
        for (Map<String,String> row : SourceInformationBenchmark.csv(cfg.getProperty("models"))) {
            int seed = Integer.parseInt(row.get("seed"));
            String v = row.get("variant");
            if (!v.equals(A) && !v.equals(B)) throw new IllegalArgumentException("Unknown variant " + v);
            t = System.nanoTime();
            SourceInformationBenchmark.Model m = new SourceInformationBenchmark.Model(row.get("modelPath"), v);
            if (SourceInformationBenchmark.models.put(seed+":"+v, m) != null)
                throw new IllegalArgumentException("Duplicate model");
            SourceInformationBenchmark.setupRow("model_load",seed,v,Double.NaN,"resident",System.nanoTime()-t,
                    m.features.length+" inputs");
        }
        for (Map<String,String> row : SourceInformationBenchmark.csv(cfg.getProperty("gates"))) {
            String key = Integer.parseInt(row.get("seed"))+":"+
                    Double.toString(SourceInformationBenchmark.number(row.get("mergeRate")))+":"+row.get("variant");
            if (SourceInformationBenchmark.gates.put(key, SourceInformationBenchmark.number(row.get("cutoff_log"))) != null)
                throw new IllegalArgumentException("Duplicate gate " + key);
        }
        List<SourceInformationBenchmark.Query> all = new ArrayList<>(cases);
        all.addAll(warmup);
        for (SourceInformationBenchmark.Query q : all) {
            for (String v : new String[]{A,B}) {
                if (!SourceInformationBenchmark.models.containsKey(q.seed+":"+v))
                    throw new IllegalArgumentException("Missing model " + q.seed+":"+v);
                SourceInformationBenchmark.cut(q,v);
            }
            if (!SourceInformationBenchmark.resident.containsKey(q.histKey())) {
                t = System.nanoTime();
                SourceInformationBenchmark.resident.put(q.histKey(), SourceInformationBenchmark.load(q));
                SourceInformationBenchmark.setupRow("histogram_context_load",-1,"all",q.rate,"resident",
                        System.nanoTime()-t,q.arm);
            }
        }
        t = System.nanoTime();
        SourceInformationBenchmark.verifyParity(cases, output,
                Double.parseDouble(cfg.getProperty("score_tolerance","1e-6")));
        SourceInformationBenchmark.setupRow("score_gate_validation",-1,"all",Double.NaN,"resident",
                System.nanoTime()-t,"all selected test cases; warms runtime before calibration warmup");
        SourceInformationBenchmark.verifyRaw(cases,warmup);
    }

    static Properties runtime(Properties cfg) {
        Properties p = new Properties();
        p.setProperty("java.version",System.getProperty("java.version"));
        p.setProperty("java.vm.name",System.getProperty("java.vm.name"));
        p.setProperty("java.vm.version",System.getProperty("java.vm.version"));
        p.setProperty("jvm.input.arguments",ManagementFactory.getRuntimeMXBean().getInputArguments().toString());
        p.setProperty("available.processors",Integer.toString(Runtime.getRuntime().availableProcessors()));
        p.setProperty("thread.cpu.supported",Boolean.toString(THREAD.isCurrentThreadCpuTimeSupported()));
        p.setProperty("thread.cpu.enabled",Boolean.toString(cpuEnabled));
        p.setProperty("launch_id",cfg.getProperty("launch_id","0"));
        p.setProperty("jvm_mode",cfg.getProperty("jvm_mode","default"));
        p.setProperty("raw.points",Integer.toString(SourceInformationBenchmark.raw.size()));
        p.setProperty("diagnostic.only","true");
        p.setProperty("routing","all arms scan raw points regardless of saved would_accept");
        p.setProperty("counter.scope","GC/JIT snapshots outside request timer; CPU samples outside exact wall timer");
        p.setProperty("warm.cache.note","selected-test verification precedes explicit calibration warmup; no cold-storage claim");
        return p;
    }

    static String required(Properties cfg, String key) {
        String value = cfg.getProperty(key);
        if (value == null || value.trim().isEmpty()) throw new IllegalArgumentException("Missing property " + key);
        return value;
    }

    public static void main(String[] args) throws Exception {
        if (args.length != 1) throw new IllegalArgumentException("Usage: FallbackCostDiagnostic CONFIG.properties");
        Properties cfg = new Properties();
        try (Reader reader = Files.newBufferedReader(Paths.get(args[0]),StandardCharsets.UTF_8)) { cfg.load(reader); }
        for (String key : new String[]{"cases","warmup","models","gates","sources","raw_files","output"}) required(cfg,key);
        int repeats = Integer.parseInt(cfg.getProperty("repeats","5"));
        int warmupRounds = Integer.parseInt(cfg.getProperty("warmup_rounds","3"));
        if (repeats < 1 || warmupRounds < 0) throw new IllegalArgumentException("Invalid repeat/warmup count");
        Path output = Paths.get(cfg.getProperty("output"));
        Files.createDirectories(output);
        for (String name : new String[]{"measurements.csv","setup.csv","parity.csv","runtime.properties"})
            if (Files.exists(output.resolve(name))) throw new IllegalArgumentException("Refusing to overwrite " + output.resolve(name));
        if (THREAD.isCurrentThreadCpuTimeSupported()) {
            try { if (!THREAD.isThreadCpuTimeEnabled()) THREAD.setThreadCpuTimeEnabled(true); }
            catch (SecurityException | UnsupportedOperationException ignored) { }
            cpuEnabled = THREAD.isThreadCpuTimeEnabled();
        }
        SourceInformationBenchmark.config = cfg;
        List<SourceInformationBenchmark.Query> cases = SourceInformationBenchmark.queries(cfg.getProperty("cases"));
        List<SourceInformationBenchmark.Query> warmup = SourceInformationBenchmark.queries(cfg.getProperty("warmup"));
        if (cases.isEmpty()) throw new IllegalArgumentException("No measured cases");
        long process0 = System.nanoTime();
        try (PrintWriter setup = new PrintWriter(Files.newBufferedWriter(output.resolve("setup.csv"),StandardCharsets.UTF_8))) {
            SourceInformationBenchmark.setup = setup;
            setup.println("stage,seed,variant,mergeRate,cache_mode,elapsed_ns,detail");
            initialize(cfg,output,cases,warmup);
            Random rng = new Random(Long.parseLong(cfg.getProperty("random_seed","0")));
            long warm0 = System.nanoTime();
            for (int round=0; round<warmupRounds; round++) {
                List<SourceInformationBenchmark.Query> order = new ArrayList<>(warmup);
                Collections.shuffle(order,rng);
                for (SourceInformationBenchmark.Query q: order) {
                    List<String> arms = new ArrayList<>(Arrays.asList(ARMS));
                    Collections.shuffle(arms,rng);
                    for (String arm: arms) validateResult(q,arm,execute(q,arm));
                }
            }
            SourceInformationBenchmark.setupRow("warmup",-1,"all",Double.NaN,"resident",System.nanoTime()-warm0,
                    warmup.size()+" calibration cases x "+warmupRounds+" rounds x 6 arms");
            Properties metadata = runtime(cfg);
            metadata.setProperty("measurement.start.epoch.millis",Long.toString(System.currentTimeMillis()));
            long iteration = 0;
            String previous = "none";
            measurementStart = System.nanoTime();
            try (PrintWriter out = new PrintWriter(Files.newBufferedWriter(output.resolve("measurements.csv"),StandardCharsets.UTF_8))) {
                out.println("launch_id,jvm_mode,seed,qId,mergeRate,repeat,block,order,previous_arm,iteration,wall_start_ns,arm,model_variant,would_accept,would_accept_a,would_accept_b,answer,truth,histogram_estimate,score_log,cutoff_log,prefix_ns,estimate_ns,features_ns,inference_ns,decision_ns,exact_ns,exact_thread_cpu_ns,total_ns,gc_count_delta,gc_time_ms_delta,compilation_time_ms_delta");
                for (int rep=0; rep<repeats; rep++) {
                    List<SourceInformationBenchmark.Query> order = new ArrayList<>(cases);
                    Collections.shuffle(order,rng);
                    for (SourceInformationBenchmark.Query q: order) {
                        List<String> arms = new ArrayList<>(Arrays.asList(ARMS));
                        Collections.shuffle(arms,rng);
                        List<String> lines = new ArrayList<>();
                        for (int position=0; position<arms.size(); position++) {
                            String arm = arms.get(position);
                            Result r = execute(q,arm);
                            validateResult(q,arm,r);
                            String v = variant(arm);
                            boolean acceptA = q.a <= SourceInformationBenchmark.cut(q,A);
                            boolean acceptB = q.b <= SourceInformationBenchmark.cut(q,B);
                            String would = v.equals("none") ? "" : Boolean.toString(r.wouldAccept);
                            lines.add(cfg.getProperty("launch_id","0")+","+cfg.getProperty("jvm_mode","default")+","+
                                    q.seed+","+q.id+","+q.rate+","+rep+","+rep+","+position+","+previous+","+(iteration++)+","+
                                    r.start+","+arm+","+v+","+would+","+acceptA+","+acceptB+","+r.answer+","+q.truth+","+
                                    r.estimateValue+","+r.score+","+r.cutoff+","+r.prefix+","+r.estimate+","+r.features+","+
                                    r.inference+","+r.decision+","+r.exact+","+r.cpu+","+r.total+","+r.gcCount+","+r.gcMillis+","+r.compilationMillis);
                            previous = arm;
                        }
                        // Output I/O happens after all six methods for this query, outside their request timers.
                        for (String line: lines) out.println(line);
                    }
                    out.flush();
                    if (out.checkError()) throw new IOException("Failed writing measurements");
                    System.out.println("Completed "+cfg.getProperty("jvm_mode","default")+" launch "+cfg.getProperty("launch_id","0")+
                            " repeat "+(rep+1)+"/"+repeats+" ("+cases.size()+" cases x 6 forced-exact arms)");
                }
            }
            metadata.setProperty("measurement.wall.ns",Long.toString(System.nanoTime()-measurementStart));
            metadata.setProperty("process.wall.ns",Long.toString(System.nanoTime()-process0));
            metadata.setProperty("measured.requests",Long.toString(iteration));
            try (Writer out = Files.newBufferedWriter(output.resolve("runtime.properties"),StandardCharsets.UTF_8)) {
                metadata.store(out,"Forced-exact diagnostic; not production gate quality or speedup");
            }
        }
    }
}

"""

## Execute

All selected queries must pass the existing score/estimate/gate parity and exact raw-count checks. Each attempt gets
a unique `fallback-cost-...` result directory. Failures and logs are preserved. No completed output is overwritten.

In [ ]:
# RUN_DIAGNOSTIC_CELL
diagnostic_config = {key: str(diag_get(key)).strip() for key in DIAG_DEFAULTS}
diagnostic_dir, diagnostic_summary, paired_summary = run_fallback_diagnostic(diagnostic_config)
display(paired_summary) if 'display' in globals() else print(paired_summary.to_string(index=False))

## Interpret the outputs

- `paired_summary.csv`: raw-scan and total paired deltas, grouped by launch, JVM mode, seed, rate, model and scan path.
  `all_queries` and `originally_rejected` are reported separately. Positive raw-scan deltas mean the scan itself
  took longer after model evaluation. The saved gate controls only subset metadata here, not routing.
- `measurements.csv`, `paired_deltas.csv`: complete requests and matched differences, including scan CPU time.
- `blocks.csv`, `order_effects.csv`: timing patterns across repetitions and predecessor/order combinations.
- `setup.csv`, per-launch `runtime.properties`, `parity.csv`, `run.json`: initialization, runtime flags and identities.
- `fallback_diagnostic_results.zip`: compact export without raw points or model weights.

Primary results retain observations with GC or compilation activity; these events are not automatically a cause.
Compare repeated JVM launches as well as blocks. Query IDs overlap across seeds and repeat across launches: no pooled
independent-query confidence interval is claimed. A persistent penalty does not prove CPU-cache causality. A penalty
that changes under shared/no-inline controls suggests runtime-path sensitivity, not an automatic application speedup.
This notebook does not revise09 production benchmark results; any runtime correction needs separate validation.

In [ ]:
# PLOT_DIAGNOSTIC_CELL
import matplotlib.pyplot as plt
table = paired_summary[paired_summary.subset == 'all_queries']
fig, axis = plt.subplots(figsize=(10, 4))
for (mode, path, variant), group in table.groupby(['jvm_mode', 'scan_path', 'variant']):
    axis.scatter(group.seed + group.launch_id * .08, group.mean_exact_delta_ms,
                 label=f'{mode} / {path} / {variant}', alpha=.8)
axis.axhline(0, color='black', linestyle='--')
axis.set(xlabel='Seed (small offset for JVM launch)', ylabel='Paired extra raw-scan time (ms)',
         title='Does model evaluation change the following raw-scan cost?')
axis.legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc='upper left')
axis.grid(alpha=.2); fig.tight_layout()
fig.savefig(Path(diagnostic_dir) / 'paired_scan_cost.png', dpi=150)
plt.show(); export_diagnostic(Path(diagnostic_dir))
print('Upload:', Path(diagnostic_dir) / 'fallback_diagnostic_results.zip')